# 3D Bound Wavepacket in a Soft-Coulomb Well

**Phase 7** of `TDSE_Solver_Plan.md`: a wavepacket in a 3D soft-Coulomb well,
`V(r) = -strength/sqrt(r^2+softening^2)` (`potentials.soft_coulomb_well`).
Unlike Phases 5-6, there's no simple closed-form trajectory to compare against
here, so the validation instead uses two things that *are* exactly predictable:

1. **Energy conservation.** `H` has no explicit time dependence, so
   `<H>=<T>+<V>` (`observables.expectation_energy`) must stay exactly constant
   under `propagator.strang_step` -- a real, non-trivial check for a bound
   3D problem, not just a restatement of Phase 1-3's checks in a new potential.
2. **A falsifiable oscillation period.** Rather than an arbitrary off-center
   Gaussian, the initial state is a superposition of the ground state and
   whichever low excited state has the strongest position-dipole overlap with
   it, both found via `stationary_states.py` (Phase 4) -- i.e. two genuinely
   different numerical methods (finite-difference diagonalization to build the
   initial state, spectral split-operator to evolve it) are tied together by a
   quantitative prediction, not just reused side by side. For
   `psi(0)=(psi0+psi1)/sqrt(2)` with real eigenstates of definite (opposite)
   parity, `<x>(t) = <psi0|x|psi1> * cos((E1-E0)*t)` exactly, in the limit that
   psi0/psi1 are exact eigenstates -- a genuine "quantum beat" whose period is
   set entirely by the energy gap. Measuring that period from the simulated
   `<x>(t)` and comparing it to `2*pi/(E1-E0)` from the independently-computed
   eigenvalues is a real cross-method check, not a tautology.

In [1]:
import sys
sys.path.insert(0, r".")
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import imageio_ffmpeg
from scipy.signal import find_peaks
from pathlib import Path

import grid as g
import propagator as prop
import potentials as pot
import observables as obs
import stationary_states as ss

matplotlib.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()
MEDIA_DIR = Path('media')
MEDIA_DIR.mkdir(exist_ok=True)


## Setup

`strength=2, softening=1` well on a `24^3` domain, `40^3` grid. `stationary_states.
lowest_states` finds the lowest 4 states: a non-degenerate s-like ground state and
a 3-fold degenerate p-like first excited manifold (same structure already seen
for the isotropic harmonic well in Phase 4). Since `eigsh` can return *any*
orthonormal basis of a degenerate subspace, the specific 3 returned p-like
vectors aren't necessarily aligned with x/y/z -- so rather than assuming one of
them is "the" x-dipole state, the code checks all 3 and picks whichever has the
largest `|<psi0|x|psi_i>|`.

In [4]:
L, N = 24.0, 80
grid3d = g.make_grid((L, L, L), (N, N, N), boundary='periodic')
strength, softening = 2.0, 1.0
V = pot.soft_coulomb_well(grid3d, strength, softening)

energies, states = ss.lowest_states(grid3d, V, k=4)
psi_gs, E0 = states[0], energies[0]
x_coord = grid3d.coords[0]

overlaps = [np.sum(psi_gs * x_coord * states[i]) for i in range(1, 4)]
best_i = 1 + int(np.argmax(np.abs(overlaps)))
psi_exc, E1 = states[best_i], energies[best_i]
dipole = np.sum(psi_gs * x_coord * psi_exc)

print(f"E0={E0:.5f}  E1={E1:.5f}  gap={E1 - E0:.5f}")
print(f"dipole matrix element <psi0|x|psi1> = {dipole:.5f}")

psi = (psi_gs + psi_exc) / np.sqrt(2)
H_initial = obs.expectation_energy(psi, grid3d, V)
print(f"initial <H> (spectral) = {H_initial:.6f}   vs finite-difference (E0+E1)/2 = {(E0 + E1) / 2:.6f}")
print("(small difference expected -- independent discretizations of the same continuum problem)")

predicted_period = 2 * np.pi / (E1 - E0)
print(f"predicted oscillation period = {predicted_period:.4f}")


E0=-0.77429  E1=-0.39133  gap=0.38296
dipole matrix element <psi0|x|psi1> = -0.85690
initial <H> (spectral) = -0.581071   vs finite-difference (E0+E1)/2 = -0.582811
(small difference expected -- independent discretizations of the same continuum problem)
predicted oscillation period = 16.4067


## Propagate and check

Run for a bit over 2 predicted periods, recording `<x>(t)` and `<H>(t)` along
the way, plus periodic density snapshots (a `z~0` slice) for the animation.

In [5]:
dt = 0.01
T_total = 2.2 * predicted_period
n_steps = round(T_total / dt)
record_every = 5
target_frames = 180
frame_stride = max(1, n_steps // target_frames)

iz0 = np.argmin(np.abs(grid3d.axes[2]))

k2 = prop.kinetic_eigenvalues(grid3d)
t_list, x_list, H_list, frames = [], [], [], []
t = 0.0
for step in range(n_steps):
    psi = prop.strang_step(psi, grid3d, V, dt, k2=k2)
    t += dt
    if step % record_every == 0:
        t_list.append(t)
        x_list.append(obs.expectation_position(psi, grid3d)[0])
        H_list.append(obs.expectation_energy(psi, grid3d, V))
    if step % frame_stride == 0:
        frames.append((t, obs.probability_density(psi)[:, :, iz0].copy()))

t_arr, x_arr, H_arr = np.array(t_list), np.array(x_list), np.array(H_list)
print(f"{len(t_arr)} samples recorded, {len(frames)} animation frames")


722 samples recorded, 181 animation frames


In [6]:
# --- Check A: energy conservation ---
H0 = H_arr[0]
max_H_drift = np.max(np.abs(H_arr - H0)) / abs(H0)
print(f"max relative <H> drift over the run: {max_H_drift:.2e}")
assert max_H_drift < 1e-4, "energy is not conserved to the expected tolerance"
print("PASS: <H> conserved (no explicit time dependence in H)")


max relative <H> drift over the run: 8.82e-08
PASS: <H> conserved (no explicit time dependence in H)


In [7]:
# --- Check B: <x>(t) oscillation period matches 2*pi/(E1-E0) ---
peaks, _ = find_peaks(x_arr)
peak_times = t_arr[peaks]
measured_period = np.mean(np.diff(peak_times))
period_rel_err = abs(measured_period - predicted_period) / predicted_period
print(f"measured period={measured_period:.4f}  predicted={predicted_period:.4f}  rel_err={period_rel_err:.4f}")
assert period_rel_err < 0.03, "oscillation period disagrees with 2*pi/(E1-E0) by more than expected"
print("PASS: <x>(t) oscillation period matches the independently-predicted 2*pi/(E1-E0)")

x_predicted = dipole * np.cos((E1 - E0) * t_arr)


measured period=16.4500  predicted=16.4067  rel_err=0.0026
PASS: <x>(t) oscillation period matches the independently-predicted 2*pi/(E1-E0)


In [8]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(t_arr, x_arr, label='simulated <x>(t)', color='C0')
axes[0].plot(t_arr, x_predicted, '--', label='dipole*cos((E1-E0)t)', color='C3')
axes[0].set_ylabel('<x>')
axes[0].legend()
axes[0].set_title('Quantum beat: dipole oscillation between ground and excited state')

axes[1].plot(t_arr, (H_arr - H0), color='C2')
axes[1].set_xlabel('t')
axes[1].set_ylabel('<H>(t) - <H>(0)')
axes[1].set_title(f'energy conservation (max relative drift = {max_H_drift:.1e})')
fig.tight_layout()
fig.savefig(MEDIA_DIR / 'bound_wavepacket_3d_checks.png', dpi=150)
plt.show()


C:\Users\Hasan's Laptop\AppData\Local\Temp\ipykernel_18212\4192300160.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
fig, ax = plt.subplots(figsize=(6, 6))
x_axis, y_axis = grid3d.axes[0], grid3d.axes[1]
extent = [x_axis.min(), x_axis.max(), y_axis.min(), y_axis.max()]
vmax = np.percentile(frames[len(frames) // 4][1], 99.9)
im = ax.imshow(frames[0][1].T, origin='lower', extent=extent, cmap='inferno', vmin=0, vmax=vmax)
title = ax.set_title('t=0.00')
ax.set_xlabel('x'); ax.set_ylabel('y')

def update(i):
    t_i, dens = frames[i]
    im.set_array(dens.T)
    title.set_text(f't={t_i:.2f}  (period={predicted_period:.1f})')
    return im, title

ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=50, blit=False)
ani.save(MEDIA_DIR / 'bound_wavepacket_3d.mp4', writer='ffmpeg', fps=24, dpi=120)
plt.close(fig)
print('saved media/bound_wavepacket_3d.mp4')


saved media/bound_wavepacket_3d.mp4
